## Part 3 Three way Testing (PDBBind)

The main file for PCRGN.This is where we will be testing all three methods Adam+weights, RMSE only and PCGrad using the physics weights that have calculated.


In [18]:
import os
print(os.path.abspath('..') + '/restu')

/home/exouser/git_test/multi-objective-pdbbind/restu


In [1]:
"""
PDBBind Three-Way Comparison — Full Dataset
Tests: PCGrad+Physics, Adam+Physics, RMSE-Only

Usage:
    python pdbbind_comparison.py --run-all        # all three
    python pdbbind_comparison.py --no-pcgrad      # Adam+Physics and RMSE-Only only
    python pdbbind_comparison.py                  # same as --no-pcgrad (default)

Includes all improvements:
- RemoveHs (uses PDBBind_full_noH.pkl if available, falls back to full with runtime removal)
- Physics normalization using train set statistics only (no leakage)
- Tuned hyperparameters from sweep (lr=5e-4, l2=1e-4, dropout=0.05, epochs=300)
- Best model checkpoint restoration before evaluation
- Subset pkl split for consistent train/test split
"""

import torch
import torch.optim as optim
import pandas as pd
import numpy as np
import sys
import os
import pickle
import time
import csv
import argparse
from datetime import timedelta, datetime
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem

sys.path.insert(0, '/home/exouser/git_test/multi-objective-pdbbind')

from models.dcFeaturizer import atom_features as get_atom_features
from models.layers_pytorch_pdbbind import PGGCNModel

sys.path.insert(0, '/home/exouser/git_test/multi-objective-pdbbind/pdbbind')
from train_split_data import load_data_with_saved_split

try:
    from models.pcgrad_pytorch import PCGrad
    PCGRAD_AVAILABLE = True
    print("✓ PCGrad optimizer available")
except ImportError:
    PCGRAD_AVAILABLE = False
    print("✗ PCGrad not available")



No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (/home/exouser/miniconda3/envs/pytorch-env/lib/python3.10/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-ge

✓ PCGrad optimizer available


## Configuration

Make sure change Line 11 SPLIT_PATH to the subset of that set previous in pareto front visualisation. Line 26 PHYSICS_WEIGHT should also be set depending on subset being used according to the following chart.

![image](../figure/physics_weights.png)

In [2]:

# ============================================================================
# CONFIG
# ============================================================================

class Config:
    currentpath = os.path.abspath('..')
    # Data — uses subset pkl for consistent split, full CSV/PKL for featurization
    CSV_PATH  = currentpath + '/Datasets/pdbbind.csv'
    PKL_PATH  = currentpath + '/Datasets/PDBBind_full_noH.pkl'
    PKL_PATH_FALLBACK = currentpath + '/Datasets/PDBBind_full.pkl'
    SPLIT_PATH = currentpath + '/Datasets/subsets/pdbbind_subset_100.pkl'

    # Model architecture
    NUM_ATOM_FEATURES = 36
    R_OUT_CHANNEL     = 20
    C_OUT_CHANNEL     = 1024
    DROPOUT_RATE      = 0.05      # tuned from sweep

    # Training hyperparameters — tuned from sweep
    EPOCHS        = 300
    BATCH_SIZE    = 8
    LEARNING_RATE = 5e-4
    L2_WEIGHT     = 1e-4
    MAX_NORM      = 3.0

    PHYSICS_WEIGHT = 0.99

    RANDOM_SEED = 50

    # Save directories
    SAVE_DIR    = currentpath + '/savedir'
    RESULTS_DIR = currentpath + '/resultsdir'



In [4]:

# ============================================================================
# DATASET
# ============================================================================

class MoleculeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def collate_molecules(batch):
    return [item[0] for item in batch], torch.FloatTensor([item[1] for item in batch])


# ============================================================================
# UTILITIES
# ============================================================================

def set_random_seeds(seed=50):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def format_time(seconds):
    return str(timedelta(seconds=int(seconds)))


def apply_maxnorm_constraint(model, max_norm=3.0):
    with torch.no_grad():
        for param in model.parameters():
            if param.requires_grad and param.dim() >= 2:
                norm = param.norm(2, dim=0, keepdim=True)
                desired = torch.clamp(norm, max=max_norm)
                param.mul_(desired / (norm + 1e-7))


def compute_task_losses(predictions, targets, physics_info, physics_weight):
    targets = targets.view(-1, 1)

    empirical_loss = torch.sqrt(torch.mean((predictions - targets) ** 2))

    host_energy    = physics_info[:, [0, 3, 6, 9, 12]].sum(dim=1, keepdim=True)
    guest_energy   = physics_info[:, [1, 4, 7, 10, 13]].sum(dim=1, keepdim=True)
    complex_energy = physics_info[:, [2, 5, 8, 11, 14]].sum(dim=1, keepdim=True)
    dG_physics = complex_energy - (host_energy + guest_energy)

    raw_physics_loss      = torch.sqrt(torch.mean((predictions - dG_physics) ** 2))
    weighted_physics_loss = physics_weight * raw_physics_loss
    mae = torch.mean(torch.abs(predictions - targets))

    return empirical_loss, weighted_physics_loss, raw_physics_loss, mae


def normalize_physics(mol_list, mean, std):
    normalized = []
    for mol in mol_list:
        mol = mol.clone()
        mol[:, -15:] = (mol[:, -15:] - mean) / std
        normalized.append(mol)
    return normalized


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data(config):
    """
    Load full PDBBind dataset using saved split.
    Handles hydrogen removal and physics normalization.
    """
    print("\n" + "-" * 80)
    print("Loading Data")
    print("-" * 80)

    # Check which PKL to use
    if os.path.exists(config.PKL_PATH):
        print(f"✓ Using preprocessed (no-H) PKL")
        remove_hs = False
        pkl_path  = config.PKL_PATH
    else:
        print(f"  noH PKL not found — using full PKL with RemoveHs at runtime")
        remove_hs = True
        pkl_path  = config.PKL_PATH_FALLBACK

    # Override PKL path in config for load_data_with_saved_split
    class _Cfg:
        CSV_PATH = config.CSV_PATH
        PKL_PATH = pkl_path

    # load_data_with_saved_split handles RemoveHs internally when using full PKL
    # but we need to check whether it's the noH version or not
    if remove_hs:
        # Load manually so we can apply RemoveHs
        X_train, X_test, y_train, y_test = _load_with_remove_hs(config, pkl_path)
    else:
        X_train, X_test, y_train, y_test = load_data_with_saved_split(_Cfg(), config.SPLIT_PATH)

    print(f"\n✓ Train: {len(X_train)} | Test: {len(X_test)}")
    print(f"\nTarget statistics (train):")
    print(f"  Mean: {np.mean(y_train):.2f} kcal/mol")
    print(f"  Std:  {np.std(y_train):.2f} kcal/mol")
    print(f"  Min:  {np.min(y_train):.2f} kcal/mol")
    print(f"  Max:  {np.max(y_train):.2f} kcal/mol")

    return X_train, X_test, y_train, y_test


def _load_with_remove_hs(config, pkl_path):
    """Fallback loader that applies RemoveHs at runtime."""
    physics_columns = [
        'pb-protein-vdwaals', 'pb-ligand-vdwaals', 'pb-complex-vdwaals',
        'gb-protein-1-4-eel', 'gb-ligand-1-4-eel', 'gb-complex-1-4-eel',
        'gb-protein-eelect',  'gb-ligand-eelec',   'gb-complex-eelec',
        'gb-protein-egb',     'gb-ligand-egb',      'gb-complex-egb',
        'gb-protein-esurf',   'gb-ligand-esurf',    'gb-complex-esurf'
    ]

    with open(config.SPLIT_PATH, 'rb') as f:
        split_data = pickle.load(f)
    with open(pkl_path, 'rb') as f:
        pdb_dict = pickle.load(f)

    df = pd.read_csv(config.CSV_PATH)
    df = df.dropna(subset=['ddg'])
    df = df[df['complex-name'].apply(lambda x: 'E+' not in str(x))]
    df = df.set_index('complex-name')

    def featurize_split(names):
        X, y = [], []
        skipped = 0
        for pdb_id in names:
            if pdb_id not in df.index or pdb_id not in pdb_dict:
                skipped += 1
                continue
            row = df.loc[pdb_id]
            info_array = row[physics_columns].tolist()
            target = row['ddg']
            try:
                mol = Chem.RemoveHs(pdb_dict[pdb_id])
                feat = []
                for atom in mol.GetAtoms():
                    base_feat = get_atom_features(atom)
                    new_feature = base_feat.tolist()
                    position = mol.GetConformer().GetAtomPosition(atom.GetIdx())
                    new_feature += [atom.GetMass(), atom.GetAtomicNum(), atom.GetFormalCharge()]
                    new_feature += [position.x, position.y, position.z]
                    neighbors = atom.GetNeighbors()[:2]
                    for neighbor in neighbors:
                        new_feature += [float(neighbor.GetIdx())]
                    for _ in range(2 - len(neighbors)):
                        new_feature += [-1.0]
                    feat.append(new_feature + info_array)
                X.append(torch.FloatTensor(np.array(feat)))
                y.append(float(target))
            except Exception:
                skipped += 1
        if skipped:
            print(f"  Warning: skipped {skipped} structures")
        return X, y

    train_names = [n for n in split_data['train_names'] if n in set(df.index)]
    test_names  = [n for n in split_data['test_names']  if n in set(df.index)]

    print("  Featurizing training set...")
    X_train, y_train = featurize_split(train_names)
    print("  Featurizing test set...")
    X_test,  y_test  = featurize_split(test_names)

    # Normalize using train stats only
    train_physics = torch.stack([mol[0, -15:] for mol in X_train])
    phys_mean = train_physics.mean(dim=0)
    phys_std  = train_physics.std(dim=0).clamp(min=1e-8)
    X_train = normalize_physics(X_train, phys_mean, phys_std)
    X_test  = normalize_physics(X_test,  phys_mean, phys_std)

    return X_train, X_test, y_train, y_test


# ============================================================================
# TRAINING
# ============================================================================

def train_model(model, train_loader, test_loader,
                use_pcgrad, use_physics, config, device, run_name):
    model = model.to(device)

    physics_weight = config.PHYSICS_WEIGHT if use_physics else 0.0

    if use_pcgrad and PCGRAD_AVAILABLE and use_physics:
        base_optimizer = optim.Adam(model.parameters(),
                                    lr=config.LEARNING_RATE,
                                    weight_decay=config.L2_WEIGHT)
        optimizer = PCGrad(base_optimizer)
        opt_name  = "PCGrad + Adam"
    else:
        optimizer = optim.Adam(model.parameters(),
                               lr=config.LEARNING_RATE,
                               weight_decay=config.L2_WEIGHT)
        opt_name  = "Adam"

    print(f"\n{'='*80}")
    print(f"Training: {run_name}")
    print(f"{'='*80}")
    print(f"  Optimizer:      {opt_name}")
    print(f"  Physics loss:   {'ENABLED (λ=' + str(physics_weight) + ')' if use_physics else 'DISABLED'}")
    print(f"  Epochs:         {config.EPOCHS}")
    print(f"  Learning rate:  {config.LEARNING_RATE}")
    print(f"  L2 weight:      {config.L2_WEIGHT}")
    print(f"  Dropout:        {config.DROPOUT_RATE}")

    best_val_mae     = float('inf')
    best_model_state = None
    start_time       = time.time()

    for epoch in range(config.EPOCHS):
        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = [x.to(device) for x in X_batch]
            y_batch = y_batch.unsqueeze(1).to(device)

            predictions, _, physics_info = model(X_batch, training=True)
            emp, phys_w, _, _ = compute_task_losses(
                predictions, y_batch, physics_info, physics_weight)

            if use_pcgrad and PCGRAD_AVAILABLE and use_physics:
                optimizer.pc_backward([emp, phys_w])
                optimizer.step()
            else:
                optimizer.zero_grad()
                total_loss = emp + phys_w if use_physics else emp
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            apply_maxnorm_constraint(model, config.MAX_NORM)

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            all_preds, all_targets, all_phys = [], [], []
            for X_batch, y_batch in test_loader:
                X_batch = [x.to(device) for x in X_batch]
                y_batch = y_batch.unsqueeze(1).to(device)
                preds, _, phys = model(X_batch, training=False)
                all_preds.append(preds)
                all_targets.append(y_batch)
                all_phys.append(phys)

            val_preds   = torch.cat(all_preds)
            val_targets = torch.cat(all_targets)
            val_phys    = torch.cat(all_phys)
            _, _, _, val_mae = compute_task_losses(
                val_preds, val_targets, val_phys, physics_weight)

        if val_mae.item() < best_val_mae:
            best_val_mae     = val_mae.item()
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 50 == 0 or epoch == 0:
            elapsed = time.time() - start_time
            eta     = (elapsed / (epoch + 1)) * (config.EPOCHS - epoch - 1)
            print(f"  Epoch {epoch+1:3d}/{config.EPOCHS} | "
                  f"Val MAE: {val_mae.item():.4f} | "
                  f"Best: {best_val_mae:.4f} | "
                  f"Time: {format_time(elapsed)} | ETA: {format_time(eta)}")

    print(f"\nRestoring best model (val MAE: {best_val_mae:.4f})")
    model.load_state_dict(best_model_state)
    print(f"Completed in {format_time(time.time() - start_time)}")

    return model, best_val_mae


# ============================================================================
# EVALUATION
# ============================================================================

def evaluate_model(model, test_loader, config, device):
    model.eval()
    all_preds, all_targets, all_phys = [], [], []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = [x.to(device) for x in X_batch]
            y_batch = y_batch.unsqueeze(1).to(device)
            preds, _, phys = model(X_batch, training=False)
            all_preds.append(preds)
            all_targets.append(y_batch)
            all_phys.append(phys)

    predictions = torch.cat(all_preds)
    targets     = torch.cat(all_targets)
    physics     = torch.cat(all_phys)

    rmse = torch.sqrt(torch.mean((predictions - targets) ** 2)).item()
    mae  = torch.mean(torch.abs(predictions - targets)).item()

    ss_res = torch.sum((targets - predictions) ** 2).item()
    ss_tot = torch.sum((targets - torch.mean(targets)) ** 2).item()
    r2     = 1 - (ss_res / ss_tot)

    _, _, raw_physics_loss, _ = compute_task_losses(
        predictions, targets, physics, physics_weight=1.0)

    return {
        'rmse':         rmse,
        'mae':          mae,
        'r2':           r2,
        'physics_loss': raw_physics_loss.item(),
    }


def print_sample_predictions(predictions_tensor, y_test, n=10):
    preds = predictions_tensor.cpu().numpy()
    print(f"\n{'True Value':>12} | {'Prediction':>12} | {'Error':>12}")
    print("-" * 42)
    for i in range(min(n, len(y_test))):
        true_val = y_test[i]
        pred_val = preds[i][0] if preds.ndim > 1 else preds[i]
        print(f"{true_val:>12.4f} | {pred_val:>12.4f} | {pred_val - true_val:>12.4f}")


# ============================================================================
# MAIN
# ============================================================================

def main():
    '''
    #Juypter seems to unable to parse arguements
    parser = argparse.ArgumentParser(
         description='PDBBind three-way comparison: PCGrad+Physics vs Adam+Physics vs RMSE-Only')
     group = parser.add_mutually_exclusive_group()
     group.add_argument('--run-all', action='store_true',
                        help='Run all three: PCGrad+Physics, Adam+Physics, RMSE-Only')
     group.add_argument('--no-pcgrad', action='store_true',
                        help='Run Adam+Physics and RMSE-Only only (default)')
     args = parser.parse_args()
    
    
    '''
 
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    config    = Config()

    print("=" * 80)
    print("PDBBIND THREE-WAY COMPARISON — FULL DATASET")
    print("=" * 80)
    print(f"Timestamp: {timestamp}")

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Device:    {device}")

    set_random_seeds(config.RANDOM_SEED)

    # Load data once — all runs reuse the same split
    X_train, X_test, y_train, y_test = load_data(config)

    train_loader = DataLoader(
        MoleculeDataset(X_train, y_train),
        batch_size=config.BATCH_SIZE, shuffle=True, collate_fn=collate_molecules)
    test_loader = DataLoader(
        MoleculeDataset(X_test, y_test),
        batch_size=config.BATCH_SIZE, shuffle=False, collate_fn=collate_molecules)

    # Define runs based on args
    '''
    if args.run_all:
            runs = [
                ('ΔG with PCGrad + Multi-loss', True,  True),
                ('ΔG with Adam + Multi-loss',   False, True),
                ('ΔG without Multi-loss',        False, False),
            ]
        else:
            # Default / --no-pcgrad
            runs = [
                ('ΔG with Adam + Multi-loss', False, True),
                ('ΔG without Multi-loss',      False, False),
            ]
    
    '''
    runs = [
            ('ΔG with Adam + Multi-loss', False, True),
            ('ΔG without Multi-loss',      False, False),
        ]

    results = []

    for run_name, use_pcgrad, use_physics in runs:
        # Fresh model for each run
        model = PGGCNModel(
            num_atom_features=config.NUM_ATOM_FEATURES,
            r_out_channel=config.R_OUT_CHANNEL,
            c_out_channel=config.C_OUT_CHANNEL,
            dropout_rate=config.DROPOUT_RATE
        )
        model.add_rule("sum", 0, 32)
        model.add_rule("multiply", 32, 33)
        model.add_rule("distance", 33, 36)

        print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

        model, best_val_mae = train_model(
            model, train_loader, test_loader,
            use_pcgrad, use_physics, config, device, run_name)

        metrics = evaluate_model(model, test_loader, config, device)

        print(f"\nResults — {run_name}:")
        print(f"  RMSE:          {metrics['rmse']:.4f} kcal/mol")
        print(f"  MAE:           {metrics['mae']:.4f} kcal/mol")
        print(f"  R²:            {metrics['r2']:.4f}")
        print(f"  Physics loss:  {metrics['physics_loss']:.4f}")
        print(f"  Best val MAE:  {best_val_mae:.4f}")

        results.append({
            'model':        run_name,
            'rmse':         metrics['rmse'],
            'mae':          metrics['mae'],
            'r2':           metrics['r2'],
            'physics_loss': metrics['physics_loss'],
            'best_val_mae': best_val_mae,
        })

        # Save model checkpoint
        os.makedirs(config.SAVE_DIR, exist_ok=True)
        suffix = 'pcgrad' if use_pcgrad else ('adam' if use_physics else 'rmse_only')
        model_path = os.path.join(
            config.SAVE_DIR,
            f'pdbbind_full_{suffix}_lr{config.LEARNING_RATE}_'
            f'pw{config.PHYSICS_WEIGHT}_{timestamp}.pth')
        torch.save({
            'model_state_dict': model.state_dict(),
            'metrics':          metrics,
            'best_val_mae':     best_val_mae,
            'timestamp':        timestamp,
            'config': {
                'learning_rate':  config.LEARNING_RATE,
                'l2_weight':      config.L2_WEIGHT,
                'dropout_rate':   config.DROPOUT_RATE,
                'physics_weight': config.PHYSICS_WEIGHT,
                'batch_size':     config.BATCH_SIZE,
                'epochs':         config.EPOCHS,
                'use_pcgrad':     use_pcgrad,
                'use_physics':    use_physics,
            }
        }, model_path)
        print(f"✓ Model saved to: {model_path}")

    # ── Final comparison table ─────────────────────────────────────────────────
    print(f"\n{'='*80}")
    print("FINAL COMPARISON TABLE")
    print(f"{'='*80}")
    print(f"\n{'Model':<35} | {'RMSE':>8} | {'MAE':>8} | {'R²':>8} | {'PhysLoss':>10}")
    print("-" * 80)
    for r in results:
        print(f"{r['model']:<35} | "
              f"{r['rmse']:>8.4f} | "
              f"{r['mae']:>8.4f} | "
              f"{r['r2']:>8.4f} | "
              f"{r['physics_loss']:>10.4f}")

    # Save results CSV
    os.makedirs(config.RESULTS_DIR, exist_ok=True)
    csv_path = os.path.join(config.RESULTS_DIR,
                            f'pdbbind_comparison_full_{timestamp}.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
    print(f"\n✓ Results saved to: {csv_path}")

    # LaTeX snippet
    print(f"\n{'='*80}")
    print("LATEX TABLE ROWS")
    print(f"{'='*80}")
    for r in results:
        print(f"    {r['model']} & "
              f"{r['rmse']:.2f} & "
              f"{r['physics_loss']:.2f} & "
              f"{r['mae']:.2f} \\\\")

    print(f"\n{'='*80}")
    print("COMPARISON COMPLETE")
    print(f"{'='*80}")


if __name__ == "__main__":
    main()


PDBBIND THREE-WAY COMPARISON — FULL DATASET
Timestamp: 20260810-172047
Device:    cuda

--------------------------------------------------------------------------------
Loading Data
--------------------------------------------------------------------------------
✓ Using preprocessed (no-H) PKL
Loading PDBBind dataset with saved split...
✓ Loaded split from: /home/exouser/git_test/multi-objective-pdbbind/Datasets/subsets/pdbbind_subset_100.pkl
  Total: 100
  Train: 80
  Test: 20


✓ Loaded 2914 CSV entries, 2836 PDB structures
✓ Using saved split: 80 train, 20 test
  Featurizing training set...
  Featurizing test set...
✓ Featurized 80 train, 20 test samples
✓ Physics normalized (max abs after: 5.9292)

✓ Train: 80 | Test: 20

Target statistics (train):
  Mean: -8.90 kcal/mol
  Std:  2.68 kcal/mol
  Min:  -14.78 kcal/mol
  Max:  -3.00 kcal/mol

Model parameters: 55,282


/home/exouser/miniconda3/envs/pytorch-env/lib/python3.10/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")



Training: ΔG with Adam + Multi-loss
  Optimizer:      Adam
  Physics loss:   ENABLED (λ=0.99)
  Epochs:         300
  Learning rate:  0.0005
  L2 weight:      0.0001
  Dropout:        0.05
  Epoch   1/300 | Val MAE: 14.3002 | Best: 14.3002 | Time: 0:00:00 | ETA: 0:02:12
  Epoch  50/300 | Val MAE: 8.1218 | Best: 5.2129 | Time: 0:00:07 | ETA: 0:00:37
  Epoch 100/300 | Val MAE: 8.1283 | Best: 5.2129 | Time: 0:00:14 | ETA: 0:00:29
  Epoch 150/300 | Val MAE: 8.1278 | Best: 5.2129 | Time: 0:00:22 | ETA: 0:00:22
  Epoch 200/300 | Val MAE: 8.1305 | Best: 5.2129 | Time: 0:00:29 | ETA: 0:00:14
  Epoch 250/300 | Val MAE: 8.1315 | Best: 5.2129 | Time: 0:00:36 | ETA: 0:00:07
  Epoch 300/300 | Val MAE: 8.1280 | Best: 5.2129 | Time: 0:00:43 | ETA: 0:00:00

Restoring best model (val MAE: 5.2129)
Completed in 0:00:43

Results — ΔG with Adam + Multi-loss:
  RMSE:          6.0235 kcal/mol
  MAE:           5.2129 kcal/mol
  R²:            -4.5615
  Physics loss:  5.0430
  Best val MAE:  5.2129
✓ Model sa